# DeepCpf1 split notebook (TXT input)
This version uses plain text files instead of a CSV.

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEQUENCE_FILE = Path("target_context_34nt.txt")
ACTIVITY_FILE = Path("activity.txt")      # one activity per line
CHROMATIN_FILE = Path("chromatin_accessibility.txt")  # optional; delete or set to None if unavailable

SPLIT_SEED=42
SELECTED_SEEN_SIZE=12000
VALIDATION_FRACTION=0.20

OUTDIR=Path("results/deepcpf1_fixed_split")
(OUTDIR/"saved_splits").mkdir(parents=True,exist_ok=True)
(OUTDIR/"split_sequences").mkdir(parents=True,exist_ok=True)


In [ ]:

with open(SEQUENCE_FILE) as f:
    sequences=[x.strip().upper() for x in f if x.strip()]

with open(ACTIVITY_FILE) as f:
    activity=[float(x.strip()) for x in f if x.strip()]

if CHROMATIN_FILE is not None and CHROMATIN_FILE.exists():
    with open(CHROMATIN_FILE) as f:
        chromatin=[int(float(x.strip())) for x in f if x.strip()]
else:
    chromatin=None

assert len(sequences)==len(activity)
if chromatin is not None:
    assert len(chromatin)==len(sequences)

for s in sequences:
    assert len(s)==34
    assert set(s)<=set("ACGT")

df=pd.DataFrame({
    "target_context_34nt":sequences,
    "activity":activity
})
if chromatin is not None:
    df["chromatin_accessibility"]=chromatin

df["sample_id"]=np.arange(len(df))
print(df.head())
print("Samples:",len(df))


In [ ]:

np.random.seed(SPLIT_SEED)

all_idx=np.arange(len(df))
selected=np.random.choice(len(df),size=SELECTED_SEEN_SIZE,replace=False)
unseen=np.setdiff1d(all_idx,selected)
train,validation=train_test_split(
    selected,
    test_size=VALIDATION_FRACTION,
    random_state=SPLIT_SEED
)

train_df=df.iloc[train].reset_index(drop=True)
validation_df=df.iloc[validation].reset_index(drop=True)
unseen_df=df.iloc[unseen].reset_index(drop=True)

train_df.to_csv(OUTDIR/"saved_splits"/"train_split.csv",index=False)
validation_df.to_csv(OUTDIR/"saved_splits"/"validation_split.csv",index=False)
unseen_df.to_csv(OUTDIR/"saved_splits"/"unseen_split.csv",index=False)

def export_subset(frame,name):
    d=OUTDIR/"split_sequences"/name
    d.mkdir(parents=True,exist_ok=True)

    frame["target_context_34nt"].to_csv(d/f"{name}_target_context_34nt.txt",index=False,header=False)
    frame["activity"].to_csv(d/f"{name}_activity.txt",index=False,header=False)

    frame["target_context_34nt"].str.slice(0,4).to_csv(d/f"{name}_upstream_4nt.txt",index=False,header=False)
    frame["target_context_34nt"].str.slice(4,8).to_csv(d/f"{name}_pam_4nt.txt",index=False,header=False)
    frame["target_context_34nt"].str.slice(8,31).to_csv(d/f"{name}_protospacer_23nt.txt",index=False,header=False)
    frame["target_context_34nt"].str.slice(31,34).to_csv(d/f"{name}_downstream_3nt.txt",index=False,header=False)

    if "chromatin_accessibility" in frame.columns:
        frame["chromatin_accessibility"].to_csv(d/f"{name}_chromatin_accessibility.txt",index=False,header=False)

    frame.to_csv(d/f"{name}_complete.tsv",sep="\t",index=False)

export_subset(train_df,"train")
export_subset(validation_df,"validation")
export_subset(unseen_df,"unseen")

print("Done.")
print(len(train_df),len(validation_df),len(unseen_df))
